# RetailMart Lakehouse

## Notebook : 05_Load_Products

### Layer
Bronze Layer

### Objective

This notebook ingests the Products dataset into the Bronze layer of the RetailMart Lakehouse.

### Pipeline Steps

- Read Raw Products Dataset
- Perform Data Profiling
- Validate Schema
- Validate Data Quality
- Execute Business Validations
- Add Audit Columns
- Load Bronze Delta Table
- Verify Data Load
- Generate Bronze Load Report

### Source

Raw Volume

### Target

retailmart.bronze.products

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
PIPELINE_NAME = "Bronze_Products"

SOURCE_FILE = RAW_PRODUCTS

TARGET_TABLE = TARGET_TABLE_PRODUCTS

RUN_ID = generate_run_id()

START_TIME = start_pipeline()

Pipeline Started : 2026-07-15 07:37:25.636074


In [0]:
raw_products = (
    spark.read
         .option("header", True)
         .csv(SOURCE_FILE)
)

display(raw_products.limit(10))

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
PROD_000001,music,66,512,5,18964,11,25,52
PROD_000002,food,40,452,4,25798,57,42,49
PROD_000003,office,69,121,5,24989,45,16,10
PROD_000004,music,46,717,4,18661,91,8,48
PROD_000005,toys,29,632,5,26347,79,32,11
PROD_000006,books,24,845,1,12830,52,31,68
PROD_000007,sports,34,563,3,12265,66,7,20
PROD_000008,books,40,247,4,13678,21,19,63
PROD_000009,garden,55,170,3,19359,80,8,27
PROD_000010,toys,34,750,1,16878,45,17,71


In [0]:
raw_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: string (nullable = true)
 |-- product_description_length: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

In [0]:
# Explicit Schema

products_schema = StructType([

    StructField("product_id", StringType(), True),

    StructField("product_category_name", StringType(), True),

    StructField("product_name_length", StringType(), True),

    StructField("product_description_length", StringType(), True),

    StructField("product_photos_qty", StringType(), True),

    StructField("product_weight_g", StringType(), True),

    StructField("product_length_cm", StringType(), True),

    StructField("product_height_cm", StringType(), True),

    StructField("product_width_cm", StringType(), True)

])

In [0]:
products_df = (
    spark.read
        .schema(products_schema)
        .option("header", True)
        .csv(SOURCE_FILE)
)

In [0]:
# Dataset Profile
dataset_profile(products_df, "Products")

Products
Rows    : 3000
Columns : 9
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: string (nullable = true)
 |-- product_description_length: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
PROD_000001,music,66,512,5,18964,11,25,52
PROD_000002,food,40,452,4,25798,57,42,49
PROD_000003,office,69,121,5,24989,45,16,10
PROD_000004,music,46,717,4,18661,91,8,48
PROD_000005,toys,29,632,5,26347,79,32,11
PROD_000006,books,24,845,1,12830,52,31,68
PROD_000007,sports,34,563,3,12265,66,7,20
PROD_000008,books,40,247,4,13678,21,19,63
PROD_000009,garden,55,170,3,19359,80,8,27
PROD_000010,toys,34,750,1,16878,45,17,71


In [0]:
expected_schema = {
    "product_id": "string",
    "product_category_name": "string",
    "product_name_length": "string",
    "product_description_length": "string",
    "product_photos_qty": "string",
    "product_weight_g": "string",
    "product_length_cm": "string",
    "product_height_cm": "string",
    "product_width_cm": "string"
}

In [0]:
schema_status = validate_schema(
    products_df,
    products_schema
)

Schema Validation Passed


In [0]:
pk_status = validate_primary_key(
    products_df,
    "product_id"
)

Total Rows : 3000
Distinct Count : 3000
Primary Key Validation Passed — (product_id)


In [0]:
total_rows = products_df.count()
print(f"Rows    : {total_rows}")
print(f"Columns : {len(products_df.columns)}")
products_df.printSchema()
display(products_df.limit(10))

Rows    : 3000
Columns : 9
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: string (nullable = true)
 |-- product_description_length: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
PROD_000001,music,66,512,5,18964,11,25,52
PROD_000002,food,40,452,4,25798,57,42,49
PROD_000003,office,69,121,5,24989,45,16,10
PROD_000004,music,46,717,4,18661,91,8,48
PROD_000005,toys,29,632,5,26347,79,32,11
PROD_000006,books,24,845,1,12830,52,31,68
PROD_000007,sports,34,563,3,12265,66,7,20
PROD_000008,books,40,247,4,13678,21,19,63
PROD_000009,garden,55,170,3,19359,80,8,27
PROD_000010,toys,34,750,1,16878,45,17,71


In [0]:
duplicate_rows = duplicate_summary(
    products_df,
    total_rows
)

Duplicate Rows : 0


In [0]:
null_summary(products_df)

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,0,0,0,0,0,0,0,0


In [0]:
# Missing category
missing_category = products_df.filter(
    F.col("product_category_name").isNull()
).count()

print(f"Missing Product Category : {missing_category}")

Missing Product Category : 0


In [0]:
# Duplicate product name 

duplicate_categories = (
    products_df
    .groupBy("product_category_name")
    .count()
    .orderBy(F.desc("count"))
)

display(duplicate_categories)

product_category_name,count
garden,233
furniture,224
books,215
health,212
toys,208
automotive,206
electronics,199
beauty,193
music,191
fashion,191


In [0]:
# Product weight
invalid_weight = products_df.filter(
    F.col("product_weight_g") <= 0
).count()

print(f"Invalid Product Weight : {invalid_weight}")

Invalid Product Weight : 0


In [0]:
# Product length
invalid_length = products_df.filter(
    F.col("product_length_cm") <= 0
).count()

print(f"Invalid Product Length : {invalid_length}")

Invalid Product Length : 0


In [0]:
# Product height
invalid_height = products_df.filter(
    F.col("product_height_cm") <= 0
).count()

print(f"Invalid Product Height : {invalid_height}")

Invalid Product Height : 0


In [0]:
# Product width
invalid_width = products_df.filter(
    F.col("product_width_cm") <= 0
).count()

print(f"Invalid Product Width : {invalid_width}")

Invalid Product Width : 0


In [0]:
products_df.select(

    F.min(F.col("product_weight_g").cast("int")).alias("Minimum Weight"),

    F.max(F.col("product_weight_g").cast("int")).alias("Maximum Weight"),

    F.avg(F.col("product_weight_g").cast("int")).alias("Average Weight")

).show()

+--------------+--------------+--------------+
|Minimum Weight|Maximum Weight|Average Weight|
+--------------+--------------+--------------+
|           122|         29996|     15257.391|
+--------------+--------------+--------------+



In [0]:
products_df.select(
    F.avg(F.col("product_name_length").cast("int")).alias("Average Name Length"),
    F.max(F.col("product_name_length").cast("int")).alias("Maximum Name Length")
).show()

+-------------------+-------------------+
|Average Name Length|Maximum Name Length|
+-------------------+-------------------+
|  44.94233333333333|                 70|
+-------------------+-------------------+



In [0]:
products_df = add_audit_columns(
    products_df,
    PIPELINE_NAME,
    RUN_ID
)

In [0]:
status, error = write_bronze_table(
    products_df,
    TARGET_TABLE
)

Bronze table written: retailmart.bronze.products


In [0]:
# Verification

bronze_df = spark.table(TARGET_TABLE)

rows_written = bronze_df.count()

print("VERIFICATION")

print(f"Rows Written : {rows_written}")

if rows_written == total_rows:
    print("Verification Successful")
else:
    print("Row Count Mismatch")

VERIFICATION
Rows Written : 3000
Verification Successful


In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_FILE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=status
)

BRONZE LOAD REPORT
Pipeline        : Bronze_Products
Run ID          : 3cfdd221-c500-4089-8cfc-ed5711c59956
Source          : /Volumes/dbacademy/default/raw/raw_products_dataset.csv
Target          : retailmart.bronze.products
Rows Read       : 3000
Rows Written    : 3000
Duplicate Rows  : 0
Start Time      : 2026-07-15 07:37:25.636074
End Time        : 2026-07-15 07:38:23.300700
Duration (sec)  : 57.66
Status          : SUCCESS


In [0]:
# Validation Summary

validation_summary = spark.createDataFrame(

[
    (
        "Schema Validation",
        schema_status
    ),

    (
        "Primary Key Validation",
        "PASS" if pk_status else "FAIL"
    ),

    (
        "Duplicate Validation",
        "PASS" if duplicate_rows == 0 else "FAIL"
    ),

    (
        "Category Validation",
        "PASS" if missing_category == 0 else "FAIL"
    )

],

["Validation","Status"]

)

display(validation_summary)

Validation,Status
Schema Validation,PASSED
Primary Key Validation,PASS
Duplicate Validation,PASS
Category Validation,PASS


# Engineering Observations

## Summary

- Product schema validated successfully.
- Primary key uniqueness verified.
- Duplicate record validation completed.
- Product category validation completed.
- Bronze Delta table created successfully.
